In [1]:
import importlib.metadata as md, json, torch, torchvision, numpy, PIL, socket, subprocess, sys
print('START environment preflight',flush=True)
if torch.cuda.device_count()!=2: raise RuntimeError('comparison requires exactly two assigned GPUs')
GPU_NAMES=[torch.cuda.get_device_name(i) for i in range(2)]
if any('T4' not in x for x in GPU_NAMES): raise RuntimeError('expected two Tesla T4 GPUs: '+repr(GPU_NAMES))
for requirement in ('scikit-learn==1.7.2','nbformat==5.10.4'):
    command=[sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps',requirement]
    print('RUN pinned active-kernel dependency install',command,flush=True)
    result=subprocess.run(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=180)
    print(result.stdout,flush=True);result.check_returncode()
import sklearn
if md.version('scikit-learn')!='1.7.2': raise RuntimeError('requires pinned scikit-learn 1.7.2; found '+md.version('scikit-learn'))
if md.version('nbformat')!='5.10.4': raise RuntimeError('requires pinned nbformat 5.10.4 for executed notebook capture; found '+md.version('nbformat'))
# Preserve Kaggle's installed Torch/TorchVision pair; never pip-upgrade GPU packages.
socket.setdefaulttimeout(90)
ENV={'python':__import__('sys').version,'interpreter':__import__('sys').executable,'torch':torch.__version__,'torchvision':torchvision.__version__,'torch_cuda':torch.version.cuda,'numpy':numpy.__version__,'scikit-learn':sklearn.__version__,'Pillow':PIL.__version__,'nbformat':md.version('nbformat'),'gpus':GPU_NAMES,'train_device':'cuda:0','device_parallelism':False}
print('Observed environment',ENV,flush=True)
if not hasattr(torchvision.models,'EfficientNet_B1_Weights'): raise RuntimeError('installed TorchVision lacks pinned EfficientNet-B1 weights enum')
print('END environment preflight',flush=True)


CELL START 0 log /kaggle/working/experiments/spi_classifier_repair/runs/notebook9_20261028_v15/cell_evidence/cell_0.log
START environment preflight
RUN pinned active-kernel dependency install ['/usr/bin/python3', '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'scikit-learn==1.7.2']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 42.7 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1

RUN pinned active-kernel dependency install ['/usr/bin/python3', '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'nbformat==5.10.4']
  Attempting uninstall: nbformat
    Found existing installation: nbformat 5.11.1
    Uninstalling nbformat-5.11.1:
      Successfully uninstalled nbformat-5.11.1

Observed environment {'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'interpreter': '/usr/bin/python3', 'torch'

In [2]:
import hashlib,json
from pathlib import Path
from parameters.artifacts import write_json
from parameters.dataset_pipeline import discover_input_root,materialize_current_300
print('START immutable input and label verification',flush=True)
INPUT_ROOT=discover_input_root('/kaggle/input',SOURCE_MANIFEST['input']['manifest_sha256'])
records,unused_features=materialize_current_300(INPUT_ROOT,RUN_ROOT/'materialized_images')
if len(records)!=300: raise RuntimeError('expected exactly 300 input records')
labels=json.loads(LABEL_PATH.read_text(encoding='utf-8'))
if len(labels)!=300: raise RuntimeError('expected exactly 300 embedded v13 labels')
INPUT_RECEIPT={'root':str(INPUT_ROOT),'manifest_sha256':hashlib.sha256((INPUT_ROOT/'input_manifest.json').read_bytes()).hexdigest(),'record_count':len(records),'embedded_labels_sha256':hashlib.sha256(LABEL_PATH.read_bytes()).hexdigest(),'label_count':len(labels),'label_source':'verified v13 teacher pseudo-labels','feature_cache_used':False}
write_json(RUN_ROOT/'input_receipt.json',INPUT_RECEIPT)
print('Input verified',INPUT_RECEIPT,flush=True)
print('END immutable input and label verification',flush=True)


CELL START 1 log /kaggle/working/experiments/spi_classifier_repair/runs/notebook9_20261028_v15/cell_evidence/cell_1.log
START immutable input and label verification
Input mount candidates [{'path': '/kaggle/input/datasets/satyapaladugu/betterscheduler-spiv2-inputs-20261027b/input_manifest.json', 'sha256': 'b633e716aa20a36d7f5581dce49bffebc260516ba5ecf78a1ec0ee83ddd92be0'}]
Input verified {'root': '/kaggle/input/datasets/satyapaladugu/betterscheduler-spiv2-inputs-20261027b', 'manifest_sha256': 'b633e716aa20a36d7f5581dce49bffebc260516ba5ecf78a1ec0ee83ddd92be0', 'record_count': 300, 'embedded_labels_sha256': '1f5b3d78124d79c6fd43bb87ba2e7e94a0c18852bc45ca020aa550b40c7c75a5', 'label_count': 300, 'label_source': 'verified v13 teacher pseudo-labels', 'feature_cache_used': False}
END immutable input and label verification
CELL END 1 seconds 5.35801742700005


In [3]:
import json,importlib.util
from pathlib import Path
print('START bounded EfficientNet-B1 transfer comparison; CPU orchestration; train on cuda:0',flush=True)
cnn_path=PROJECT/'experiments/spi_classifier_repair/cnn_compare.py'
cnn_spec=importlib.util.spec_from_file_location('spi_cnn_compare',cnn_path)
cnn_compare=importlib.util.module_from_spec(cnn_spec)
cnn_spec.loader.exec_module(cnn_compare)
comparison_root=RUN_ROOT/'efficientnet_b1'
cnn_compare.run(records,labels,comparison_root,device='cuda:0',seconds=1500,max_epochs=20)
metrics=json.loads((comparison_root/'validation.json').read_text())
print('CNN validation summary',metrics,flush=True)
print('END EfficientNet-B1 comparison',flush=True)


CELL START 2 log /kaggle/working/experiments/spi_classifier_repair/runs/notebook9_20261028_v15/cell_evidence/cell_2.log
START bounded EfficientNet-B1 transfer comparison; CPU orchestration; train on cuda:0
Downloading: "https://download.pytorch.org/models/efficientnet_b1-c27df63c.pth" to /kaggle/working/project/models/torchhub/checkpoints/efficientnet_b1-c27df63c.pth

  0%|          | 0.00/30.1M [00:00<?, ?B/s]
 63%|██████▎   | 19.0M/30.1M [00:00<00:00, 198MB/s]
100%|██████████| 30.1M/30.1M [00:00<00:00, 214MB/s]
CNN epoch 1 trainF1 0.5695575901761469 innerF1 0.3538819875776397
CNN epoch 2 trainF1 0.776329389364885 innerF1 0.4318524096385542
CNN epoch 3 trainF1 0.7561587405171711 innerF1 0.4464935930780333
CNN epoch 4 trainF1 0.7692098662207358 innerF1 0.45270557615758855
CNN epoch 5 trainF1 0.7606329861015996 innerF1 0.47325824957403906
CNN epoch 6 trainF1 0.7772586890233949 innerF1 0.4974581724581724
CNN epoch 7 trainF1 0.7833562119748505 innerF1 0.48874605124605125
CNN epoch 8 train